Import libraries

In [133]:
import pandas as pd
import numpy as np
import pvlib

from pvlib.location import Location

Station07 information

In [134]:
station07_info = {
    "latitude": 36.64403,
    "longitude": 113.64187,
    "number_of_panels": 80000,
    "panel_wattage": 250,
    "tilt": 31,
    "azimuth": 180,
    "timezone": "Asia/Shanghai"
}

station07_info

{'latitude': 36.64403,
 'longitude': 113.64187,
 'number_of_panels': 80000,
 'panel_wattage': 250,
 'tilt': 31,
 'azimuth': 180,
 'timezone': 'Asia/Shanghai'}

Create timestamps

In [135]:
timestamps_df = pd.read_csv("station07_timestamps.csv")

times = pd.DatetimeIndex(
    pd.to_datetime(timestamps_df["date_time"])
)

# Make the timestamps explicitly UTC
if times.tz is None:
    times = times.tz_localize("UTC")
else:
    times = times.tz_convert("UTC")

In [136]:
print(times[:5])
print(times.tz)
print(len(times))

DatetimeIndex(['2018-06-30 16:00:00+00:00', '2018-06-30 16:15:00+00:00',
               '2018-06-30 16:30:00+00:00', '2018-06-30 16:45:00+00:00',
               '2018-06-30 17:00:00+00:00'],
              dtype='datetime64[us, UTC]', name='date_time', freq=None)
UTC
32928


In [137]:
print(times[:5])
print(times[-5:])
print(len(times))

DatetimeIndex(['2018-06-30 16:00:00+00:00', '2018-06-30 16:15:00+00:00',
               '2018-06-30 16:30:00+00:00', '2018-06-30 16:45:00+00:00',
               '2018-06-30 17:00:00+00:00'],
              dtype='datetime64[us, UTC]', name='date_time', freq=None)
DatetimeIndex(['2019-06-13 14:45:00+00:00', '2019-06-13 15:00:00+00:00',
               '2019-06-13 15:15:00+00:00', '2019-06-13 15:30:00+00:00',
               '2019-06-13 15:45:00+00:00'],
              dtype='datetime64[us, UTC]', name='date_time', freq=None)
32928


Get clear-sky, GHI-DHI-DNI. i am using Ineichen clear-sky model instead of McClear because it doesn't require an external account.

In [138]:
location = Location(
    latitude=station07_info["latitude"],
    longitude=station07_info["longitude"],
    tz=station07_info["timezone"]
)

clear_sky = location.get_clearsky(
    times,
    model="ineichen"
)

clear_sky.head()

,ghi,dni,dhi
date_time,,,
2018-06-30 16:00:00+00:00,0.0,0.0,0.0
2018-06-30 16:15:00+00:00,0.0,0.0,0.0
2018-06-30 16:30:00+00:00,0.0,0.0,0.0
2018-06-30 16:45:00+00:00,0.0,0.0,0.0
2018-06-30 17:00:00+00:00,0.0,0.0,0.0


For daytime hours:

In [139]:
clear_sky.describe()

,ghi,dni,dhi
count,32928.000000,32928.000000,32928.000000
mean,237.259580,357.072606,34.443897
std,314.882751,401.234681,46.319086
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.008928,0.161924,0.007129
75%,494.238490,817.361925,64.635965
max,986.147201,1007.562995,153.112448


In [140]:
print(clear_sky.head())
print(clear_sky[["ghi", "dni", "dhi"]].max())

                           ghi  dni  dhi
date_time                               
2018-06-30 16:00:00+00:00  0.0  0.0  0.0
2018-06-30 16:15:00+00:00  0.0  0.0  0.0
2018-06-30 16:30:00+00:00  0.0  0.0  0.0
2018-06-30 16:45:00+00:00  0.0  0.0  0.0
2018-06-30 17:00:00+00:00  0.0  0.0  0.0
ghi     986.147201
dni    1007.562995
dhi     153.112448
dtype: float64


Calculate solar position, I need the sun's position for the Perez model. This gives us things like: solar zenith, solar azimuth

In [141]:
solar_position = location.get_solarposition(times)

solar_position.head()

,apparent_zenith,zenith,apparent_elevation,elevation,azimuth,equation_of_time
date_time,,,,,,
2018-06-30 16:00:00+00:00,119.822157,119.822157,-29.822157,-29.822157,352.278638,-3.695186
2018-06-30 16:15:00+00:00,120.123723,120.123723,-30.123723,-30.123723,356.243480,-3.697214
2018-06-30 16:30:00+00:00,120.216974,120.216974,-30.216974,-30.216974,0.230081,-3.699241
2018-06-30 16:45:00+00:00,120.100922,120.100922,-30.100922,-30.100922,4.215441,-3.701268
2018-06-30 17:00:00+00:00,119.776795,119.776795,-29.776795,-29.776795,8.176612,-3.703295


Calculate extraterrestrial DNI
Perez requires dni_extra. 

In [142]:
dni_extra = pvlib.irradiance.get_extra_radiation(times)

dni_extra.head()

date_time
2018-06-30 16:00:00+00:00    1320.589024
2018-06-30 16:15:00+00:00    1320.589024
2018-06-30 16:30:00+00:00    1320.589024
2018-06-30 16:45:00+00:00    1320.589024
2018-06-30 17:00:00+00:00    1320.589024
Name: date_time, dtype: float64

Perez transposition
We take: GHI, DNI, DHI and convert them into irradiance hitting Station 07's tilted panels: POA	​


In [143]:
poa = pvlib.irradiance.get_total_irradiance(
    surface_tilt=station07_info["tilt"],
    surface_azimuth=station07_info["azimuth"],

    solar_zenith=solar_position["apparent_zenith"],
    solar_azimuth=solar_position["azimuth"],

    dni=clear_sky["dni"],
    ghi=clear_sky["ghi"],
    dhi=clear_sky["dhi"],

    dni_extra=dni_extra,

    model="perez"
)

poa.head()


,poa_global,poa_direct,poa_diffuse,poa_sky_diffuse,poa_ground_diffuse
date_time,,,,,
2018-06-30 16:00:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 16:15:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 16:30:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 16:45:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 17:00:00+00:00,0.0,0.0,0.0,0.0,0.0


This is the total clear-sku irradiance reaching the panel surface

In [144]:
poa["poa_global"]

date_time
2018-06-30 16:00:00+00:00    0.0
2018-06-30 16:15:00+00:00    0.0
2018-06-30 16:30:00+00:00    0.0
2018-06-30 16:45:00+00:00    0.0
2018-06-30 17:00:00+00:00    0.0
                            ... 
2019-06-13 14:45:00+00:00    0.0
2019-06-13 15:00:00+00:00    0.0
2019-06-13 15:15:00+00:00    0.0
2019-06-13 15:30:00+00:00    0.0
2019-06-13 15:45:00+00:00    0.0
Name: poa_global, Length: 32928, dtype: float64

Calculate Station 07 theoretical DC power: capacity then power

In [145]:
total_capacity = (
    station07_info["number_of_panels"]
    * station07_info["panel_wattage"]
)

print(f"Station 07 capacity: {total_capacity / 1e6:.2f} MW")
dc_power = (
    total_capacity
    * poa["poa_global"]
    / 1000
)

dc_power = dc_power.clip(lower=0)

Station 07 capacity: 20.00 MW


Converting DC to AC assuming a fixed effeciency for the inverter, but we are not allowed to exceed 20 000 (max Station07 capacity)

In [146]:
dc_power = (
    total_capacity
    * poa["poa_global"]
    / 1000
)

dc_power = dc_power.clip(lower=0)
inverter_efficiency = 0.98

theoretical_power = (
    dc_power * inverter_efficiency
)
theoretical_power = np.minimum(
    theoretical_power,
    total_capacity
)

In [147]:
print(
    f"Maximum theoretical power: "
    f"{theoretical_power.max() / 1e6:.3f} MW"
)

Maximum theoretical power: 20.000 MW


Everything together

In [148]:
station07_theoretical = pd.DataFrame(
    index=times
)

station07_theoretical["GHI_clear"] = (
    clear_sky["ghi"]
)

station07_theoretical["DNI_clear"] = (
    clear_sky["dni"]
)

station07_theoretical["DHI_clear"] = (
    clear_sky["dhi"]
)

station07_theoretical["POA_clear"] = (
    poa["poa_global"]
)

station07_theoretical["P_theoretical_W"] = (
    theoretical_power
)



station07_theoretical.head(20)

,GHI_clear,DNI_clear,DHI_clear,POA_clear,P_theoretical_W
date_time,,,,,
2018-06-30 16:00:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 16:15:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 16:30:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 16:45:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 17:00:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 17:15:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 17:30:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 17:45:00+00:00,0.0,0.0,0.0,0.0,0.0
2018-06-30 18:00:00+00:00,0.0,0.0,0.0,0.0,0.0


Cheking everything

In [149]:
station07_theoretical[
    [
        "GHI_clear",
        "DNI_clear",
        "DHI_clear",
        "POA_clear",
        "P_theoretical_W"
    ]
].describe()

,GHI_clear,DNI_clear,DHI_clear,POA_clear,P_theoretical_W
count,32928.000000,32928.000000,32928.000000,32928.000000,3.292800e+04
mean,237.259580,357.072606,34.443897,293.683437,5.748727e+06
std,314.882751,401.234681,46.319086,380.164093,7.436627e+06
min,0.000000,0.000000,0.000000,0.000000,0.000000e+00
25%,0.000000,0.000000,0.000000,0.000000,0.000000e+00
50%,0.008928,0.161924,0.007129,0.006956,1.363454e+02
75%,494.238490,817.361925,64.635965,658.594275,1.290845e+07
max,986.147201,1007.562995,153.112448,1066.763204,2.000000e+07


In [150]:
print(
    "Maximum theoretical power:",
    station07_theoretical["P_theoretical_W"].max(),
    "W"
)

Maximum theoretical power: 20000000.0 W


In [151]:
station07_theoretical.to_csv(
    "station07_theoretical.csv"
)